## Import Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, lower, when, datediff, unix_timestamp

## Reading from bronze layer

In [0]:
df = spark.table("olist.bronze.orders")
df.display()

## Overview about orders table

In [0]:
# Table info
print("=== Schema ===")
df.printSchema()

print("=== Row Count ===")
print(f"Total rows: {df.count()}")

print("=== Null Counts per Column ===")
df.select([
    F.count(F.when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()



## Transformations

### 1. Trim all whitespaces from string columns

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
        

### 2. Normalize improperly represented nulls in string columns 

In [0]:
NULL_STRINGS = ["", "null", "none", "n/a", "na", "unknown", "-", " "]

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(
            field.name,
            when(lower(trim(col(field.name))).isin(NULL_STRINGS), None)
            .otherwise(col(field.name))
        )

### 3. Standardize "order_status" to lowercase

In [0]:
df = df.withColumn(
    "order_status",
    lower(trim(col("order_status")))
)

### 4. Validate timestamp logical ordering (flag invalid rows)

In [0]:
# orders must be follow a logical timeline: purchase -> approve -> deliver -> customer delivery, and estimated delivery
# check each date is either NULL (not yet reached that stage) or >= purchase date
# it's fine if the date is missing OR if it comes after the purchase date
df = df.withColumn(
    "has_valid_timeline",
    (
        (col("order_approved_at").isNull() | (col("order_approved_at") >= col("order_purchase_timestamp"))) &
        (col("order_delivered_carrier_date").isNull() | (col("order_delivered_carrier_date") >= col("order_purchase_timestamp"))) &
        (col("order_delivered_customer_date").isNull() | (col("order_delivered_customer_date") >= col("order_purchase_timestamp"))) &
        (col("order_estimated_delivery_date").isNull() | (col("order_estimated_delivery_date") >= col("order_purchase_timestamp")))
    )
)  

### 5. Derive useful analytical columns

In [0]:
# delivery_time_days: total days from purchase to customer receiving the order
# approval_time_hours: how quickly the order was approved after purchase (in hours)
# estimated_vs_actual_days: negative = delivered early, positive = delivered late
# is_delivered_late: three-way logic —
#   True  = delivered after estimated date
#   False = delivered on or before estimated date
#   None  = not yet delivered (can't determine lateness)
df = df.withColumn(
    "delivery_time_days",
    F.datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
).withColumn(
    "approval_time_hours",
    (F.unix_timestamp("order_approved_at") - F.unix_timestamp("order_purchase_timestamp")) / 3600
).withColumn(
    "estimated_vs_actual_days",
    F.datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date"))
).withColumn(
    "is_delivered_late",
    when(col("order_delivered_customer_date") > col("order_estimated_delivery_date"), True)
    .when(col("order_delivered_customer_date").isNotNull(), False)
    .otherwise(None)
)

### 6. Add partition-friendly date columns

In [0]:
# Extract date components for efficient filtering and Gold layer aggregations
# order_purchase_date: useful for daily trend analysis
# year/month: useful for monthly revenue reports and Delta table partitioning
df = df.withColumn("order_purchase_date", F.to_date(col("order_purchase_timestamp"))) \
       .withColumn("order_purchase_year", F.year(col("order_purchase_timestamp"))) \
       .withColumn("order_purchase_month", F.month(col("order_purchase_timestamp")))

### 7. Handle nulls — filter out rows missing critical keys

In [0]:
df = df.filter(
    col("order_id").isNotNull() &
    col("customer_id").isNotNull() &
    col("order_purchase_timestamp").isNotNull()
)

### 8. Remove duplicate "order_id" rows

In [0]:
df = df.dropDuplicates(["order_id"])

## Quality Checks

In [0]:
print(f"Total rows after cleaning: {df.count()}")
print(f"Unique orders: {df.select('order_id').distinct().count()}")
print(f"Distinct order_status: {df.select('order_status').distinct().count()}")
print(f"Rows with invalid timeline: {df.filter(~col('has_valid_timeline')).count()}")
print(f"Null order_purchase_timestamp: {df.filter(col('order_purchase_timestamp').isNull()).count()}")
print(f"Delivered late: {df.filter(col('is_delivered_late') == True).count()}")
print(f"Delivered on time: {df.filter(col('is_delivered_late') == False).count()}")
print(f"Not yet delivered: {df.filter(col('is_delivered_late').isNull()).count()}")
df.display()

## Write to silver layer

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("olist.silver.orders")